# 06 - Automatic pipeline: tune engine -> train/predict -> reference-free head -> board-robust conformal -> bootstrap verdict
Runtime: **T4 GPU**. Run all cells top to bottom. It is **resume-safe**: re-run the same cells after a disconnect and it continues (finished detectors are skipped, predictions are cached in Drive).

**Needs:** 02 v2 done (prepared_v2 + manifest). Real-only fold detectors from 04 are reused if they exist; missing ones are trained automatically (`TRAIN_MISSING=True`).

**What it does, automatically**
1. **Engine v2:** box-localised net split/merge with alignment tolerance. Parameters tuned on boards of folds 0-1, honestly reported on held-out boards of folds 2-3.
2. **Detectors:** trains any missing fold detectors (real and/or synthetic with balanced open/short) and builds out-of-fold predictions.
3. **Reference-free head:** a small CNN that predicts open/short/none from the **test-image crop only**, trained on the template-based engine's labels (no template at inference).
4. **Board-robust conformal:** leave-one-board-out evaluation (11 boards). Compares pooled split-conformal with a worst-board variant, for four scores: raw confidence, image-only, +graph (template), +head (reference-free).
5. **Bootstrap over boards** and an automatic PASS/FAIL checklist saved to Drive.

The checklist is evidence for or against the idea, not a patentability opinion.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import sys, os
D = '/content/drive/MyDrive/pcbrisk'
sys.path.insert(0, f'{D}/code')
from pcbrisk_utils import *
seed_all(42)

sh('pip -q install ultralytics scikit-image opencv-python-headless scipy scikit-learn pandas pyyaml')
import numpy as np, pandas as pd, cv2, json, time, shutil, itertools, warnings, yaml
from scipy import ndimage as ndi
from skimage.morphology import remove_small_objects
from pathlib import Path
warnings.filterwarnings('ignore')
T0 = time.time()

Mounted at /content/drive


In [2]:
# ---------------- CONFIG (edit here) ----------------
CONFIGS = ['real']        # 'real' and/or 'synth1' (synthetic-trained detectors, balanced open/short, ratio 1.0)
SEEDS = [42]
TRAIN_MISSING = True      # train missing fold detectors automatically
MAX_TRAIN_HOURS = 3.0     # stop training gracefully after this; re-run the cell to continue
EPOCHS, IMGSZ, BATCH = 60, 640, 16
COPPER = 'auto'           # must match notebook 03 ('dark' | 'light' | 'auto')
RETUNE = False            # True forces engine re-tuning even if saved params exist
GRID_VERSION = 'g2'
MIN_AREA = 200
CRIT = [0, 1]             # open, short
NAMES = ['open', 'short', 'mousebite', 'spur', 'copper', 'pin-hole']
ALPHAS = [0.05, 0.10]
N_BOOT = 2000
CROP, HEAD_EPOCHS = 48, 8

In [3]:
restore_tar(f'{D}/01_prepared/prepared_v2.tar', f'{L}/data', marker='.ready_prepared_v2')
if not os.path.exists(f'{L}/raw/DeepPCB'):
    sh(f'unzip -q {D}/00_raw/deeppcb_main.zip -d {L}/raw')
mp = f'{D}/01_prepared/split_manifest_v2.csv'
assert sha256_file(mp) == open(f'{D}/01_prepared/manifest_v2.sha256').read(), 'manifest changed'
man = pd.read_csv(mp, dtype={'image_id': str})
P = Path(f'{L}/raw/DeepPCB/PCBData'); Y = Path(f'{L}/data/yolo_all')
temp_path = {p.name.replace('_temp.jpg', ''): p for p in P.glob('group*/*/*_temp.jpg')}
board_of = dict(zip(man.image_id, man.source_board)); fold_of = dict(zip(man.image_id, man.fold))
boards = sorted(man.source_board.unique()); assert len(boards) == 11
order = [str(b) for b in np.random.default_rng(42).permutation(boards)]      # same order as notebook 02

def load_gt(stem):
    rows = []
    for l in open(Y/'labels'/f'{stem}.txt'):
        v = l.split()
        if len(v) < 5: continue
        c, cx, cy, w, h = int(v[0]), *map(float, v[1:5])
        rows.append([c, (cx-w/2)*640, (cy-h/2)*640, (cx+w/2)*640, (cy+h/2)*640])
    return np.array(rows, float).reshape(-1, 5)
GT = {s: load_gt(s) for s in man.image_id}
print(len(GT), 'images,', sum(len(v) for v in GT.values()), 'GT boxes,', len(boards), 'boards')

1500 images, 10013 GT boxes, 11 boards


## 1. Engine v2 (template-based graph consequence) + automatic tuning

In [4]:
def copper_mask(img, copper=COPPER):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, t = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = t > 0
    if copper == 'dark' or (copper == 'auto' and m.mean() > 0.5):
        m = ~m
    return remove_small_objects(m, 30)

S8 = np.ones((3, 3)); KER = {t: np.ones((2*t+1, 2*t+1), np.uint8) for t in range(4)}

def masks(stem):
    return copper_mask(cv2.imread(str(temp_path[stem]))), copper_mask(cv2.imread(str(Y/'images'/f'{stem}.jpg')))

def prep(mt, mx, tol, sig):
    mtd = cv2.dilate(mt.astype(np.uint8), KER[tol]).astype(bool) if tol else mt
    mxd = cv2.dilate(mx.astype(np.uint8), KER[tol]).astype(bool) if tol else mx
    lt, nt = ndi.label(mt, structure=S8); lx, nx = ndi.label(mx, structure=S8)
    big = np.bincount(lt.ravel(), minlength=nt + 1) >= MIN_AREA; big[0] = False
    pair = np.bincount(lt.ravel().astype(np.int64) * (nx + 1) + lx.ravel(), minlength=(nt + 1) * (nx + 1)).reshape(nt + 1, nx + 1)
    pair[:, 0] = 0; pair[0, :] = 0
    return dict(mt=mt, mx=mx, mtd=mtd, mxd=mxd, lt=lt, lx=lx, sig=pair >= sig, big=big)

def query2(a, box, pad, minpx):
    # box-LOCALISED: missing/excess copper (beyond alignment tolerance) inside the box, and the split/merge must be visible near the box
    padl = 3 * pad
    x1, y1, x2, y2 = [int(round(v)) for v in box]
    win = lambda p: (slice(max(0, y1-p), min(640, y2+p+1)), slice(max(0, x1-p), min(640, x2+p+1)))
    w1, w2 = win(pad), win(padl)
    missing = a['mt'][w1] & ~a['mxd'][w1]; excess = a['mx'][w1] & ~a['mtd'][w1]
    nm, ne = int(missing.sum()), int(excess.sum())
    g_open = g_short = False
    if nm >= minpx:
        for n in np.unique(a['lt'][w1][missing]):
            if n == 0 or not a['big'][n]: continue
            pcs = np.unique(a['lx'][w2][a['lt'][w2] == n]); pcs = pcs[pcs > 0]
            if a['sig'][n, pcs].sum() >= 2: g_open = True; break
    if ne >= minpx:
        for j in np.unique(a['lx'][w1][excess]):
            if j == 0: continue
            nets = np.unique(a['lt'][w2][a['lx'][w2] == j]); nets = nets[nets > 0]
            if (a['sig'][nets, j] & a['big'][nets]).sum() >= 2: g_short = True; break
    return g_open, g_short, nm, ne

def verdict(go, gs, nm, ne):          # 0 none, 1 open, 2 short ('both' resolved by larger evidence)
    if go and gs: return 1 if nm >= ne else 2
    return 1 if go else (2 if gs else 0)

def eval_params(cache, params):
    tol, sig, pad, minpx = params; rows = []
    for stem, (mt, mx) in cache.items():
        a = prep(mt, mx, tol, sig)
        for c, *bx in GT[stem]:
            rows.append((int(c), verdict(*query2(a, bx, pad, minpx))))
    r = np.array(rows)
    ao = (r[r[:, 0] == 0, 1] == 1).mean(); as_ = (r[r[:, 0] == 1, 1] == 2).mean(); ff = (r[~np.isin(r[:, 0], CRIT), 1] > 0).mean()
    return ao, as_, ff

In [6]:
pj = f'{D}/05_results/engine_v2_params.json'
J = json.load(open(pj)) if os.path.exists(pj) else {}
if J.get('grid') == GRID_VERSION and not RETUNE:
    ENG = tuple(J['params'][k] for k in ['tol', 'sig', 'pad', 'minpx'])
    ao, as_, ff = J['heldout']['acc_open'], J['heldout']['acc_short'], J['heldout']['false_flag']
    print('loaded saved engine params (set RETUNE=True to redo):', dict(zip(['tol', 'sig', 'pad', 'minpx'], ENG)))
else:
    rng = np.random.default_rng(0)
    tune_stems = list(rng.choice(man[man.fold.isin([0, 1])].image_id.values, 300, replace=False))
    held_stems = list(rng.choice(man[man.fold.isin([2, 3])].image_id.values, 300, replace=False))
    cache_t = {s: masks(s) for s in tune_stems}; cache_h = {s: masks(s) for s in held_stems}
    grid = list(itertools.product([0, 1], [10], [2, 4, 6], [1, 2, 4]))      # tol, sig, pad, minpx  (sig had no effect before)
    res = []
    for p in grid:
        a_o, a_s, f_f = eval_params(cache_t, p)
        res.append(dict(tol=p[0], sig=p[1], pad=p[2], minpx=p[3], acc_open=a_o, acc_short=a_s, false_flag=f_f, obj=(a_o + a_s) / 2 + 0.5 * (1 - f_f)))
    tr = pd.DataFrame(res).sort_values('obj', ascending=False)
    print(tr.head(6).round(3).to_string(index=False))
    best = tr.iloc[0]
    ENG = (int(best['tol']), int(best['sig']), int(best['pad']), int(best['minpx']))
    ao, as_, ff = eval_params(cache_h, ENG)
    print(f'\nTUNED {dict(zip(["tol", "sig", "pad", "minpx"], ENG))}')
    json.dump(dict(grid=GRID_VERSION, params=dict(zip(['tol', 'sig', 'pad', 'minpx'], ENG)),
                   heldout=dict(acc_open=ao, acc_short=as_, false_flag=ff)), open(pj, 'w'))
    del cache_t, cache_h
print(f'HELD-OUT boards (folds 2-3):  strict acc open={ao:.3f}  short={as_:.3f}  false-flag on non-critical={ff:.3f}')

 tol  sig  pad  minpx  acc_open  acc_short  false_flag   obj
   1   10    2      2     0.779      0.778       0.030 1.264
   1   10    2      1     0.779      0.778       0.030 1.264
   1   10    2      4     0.779      0.775       0.029 1.263
   0   10    2      1     0.777      0.778       0.032 1.261
   0   10    2      2     0.777      0.778       0.032 1.261
   0   10    2      4     0.777      0.778       0.032 1.261

TUNED {'tol': 1, 'sig': 10, 'pad': 2, 'minpx': 2}
HELD-OUT boards (folds 2-3):  strict acc open=0.644  short=0.737  false-flag on non-critical=0.012


In [7]:
# full-dataset validation with tuned params (reported, includes tuning boards)
rows = []
for stem in man.image_id:
    g = GT[stem]
    if len(g) == 0: continue
    mt, mx = masks(stem); a = prep(mt, mx, ENG[0], ENG[1])
    for c, *bx in g:
        rows.append(dict(stem=stem, cls=int(c), v=verdict(*query2(a, bx, ENG[2], ENG[3]))))
va = pd.DataFrame(rows); va['verdict'] = va.v.map({0: 'none', 1: 'open', 2: 'short'})
print(pd.crosstab(va.cls.map(dict(enumerate(NAMES))), va.verdict, normalize='index').round(3).reindex(NAMES))
va.to_csv(f'{D}/05_results/engine_v2_validation.csv', index=False)

verdict     none   open  short
cls                           
open       0.162  0.722  0.116
short      0.080  0.161  0.759
mousebite  0.983  0.008  0.009
spur       0.964  0.002  0.034
copper     0.998  0.001  0.001
pin-hole   0.991  0.007  0.002


## 2. Detectors (auto-train missing folds) + out-of-fold predictions

In [8]:
from ultralytics import YOLO
import torch

def exp_name(cfg, fold, seed):
    return f'det_yolo11n_deeppcb{"+synth1" if cfg == "synth1" else ""}_f{fold}_s{seed}_v2'

def build_split(fold, use_synth, seed):
    restore_tar(f'{D}/01_prepared/prepared_v2.tar', f'{L}/data', marker='.ready_prepared_v2')
    m = man.copy(); m['split'] = np.where(m.fold == fold, 'test', np.where(m.fold == (fold + 1) % 4, 'val', 'train'))
    S = Path(f'{L}/data/split'); shutil.rmtree(S, ignore_errors=True)
    for s in ['train', 'val', 'test']:
        (S/'images'/s).mkdir(parents=True); (S/'labels'/s).mkdir(parents=True)
    for r in m.itertuples():
        shutil.copy2(Y/'images'/f'{r.image_id}.jpg', S/'images'/r.split/f'{r.image_id}.jpg')
        shutil.copy2(Y/'labels'/f'{r.image_id}.txt', S/'labels'/r.split/f'{r.image_id}.txt')
    if use_synth:
        restore_tar(f'{D}/02_synth/synth_v2.tar', f'{L}/data', marker='.ready_synth_v2')
        sm = pd.read_csv(f'{L}/data/synth/synth_meta.csv', dtype={'template': str})
        sm = sm[sm.source_board.isin(set(m[m.split == 'train'].source_board))]
        n = int((m.split == 'train').sum()); k = min(n // 2, sm.op.value_counts().min())
        sm = sm.groupby('op').sample(k, random_state=seed)                      # balanced open/short
        for sid in sm.synth_id:
            shutil.copy2(f'{L}/data/synth/images/{sid}.jpg', S/'images'/'train'/f'{sid}.jpg')
            shutil.copy2(f'{L}/data/synth/labels/{sid}.txt', S/'labels'/'train'/f'{sid}.txt')
    yaml.safe_dump(dict(path=str(S), train='images/train', val='images/val', test='images/test',
                        names={i: n for i, n in enumerate(NAMES)}), open(f'{L}/data/data.yaml', 'w'))

def train_one(cfg, fold, seed):
    use_synth = cfg == 'synth1'; exp = exp_name(cfg, fold, seed)
    RD = f'{D}/04_runs/{exp}'; LOCAL = f'{L}/runs/{exp}'
    if os.path.exists(f'{RD}/DONE'): return
    os.makedirs(f'{RD}/weights', exist_ok=True)
    build_split(fold, use_synth, seed)
    last = {'t': 0}
    def mirror(trainer):
        if time.time() - last['t'] > 120:
            for f in ['weights/last.pt', 'weights/best.pt', 'results.csv', 'args.yaml']:
                p = os.path.join(trainer.save_dir, f)
                if os.path.exists(p): safe_copy(p, f'{RD}/{f}', verify=False)
            last['t'] = time.time()
    src, dst = f'{RD}/weights/last.pt', f'{LOCAL}/weights/last.pt'
    if os.path.exists(src) and not os.path.exists(dst): safe_copy(src, dst, verify=False)
    if os.path.exists(dst):
        mdl = YOLO(dst); mdl.add_callback('on_model_save', mirror); mdl.train(resume=True)
    else:
        mdl = YOLO('yolo11n.pt'); mdl.add_callback('on_model_save', mirror)
        mdl.train(data=f'{L}/data/data.yaml', epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=30, seed=seed,
                  deterministic=True, workers=min(4, os.cpu_count()), project=f'{L}/runs', name=exp, exist_ok=True, cache='disk')
    for f in ['weights/last.pt', 'weights/best.pt', 'results.csv', 'args.yaml']:
        if os.path.exists(f'{LOCAL}/{f}'): safe_copy(f'{LOCAL}/{f}', f'{RD}/{f}')
    r = YOLO(f'{LOCAL}/weights/best.pt').val(data=f'{L}/data/data.yaml', split='test', imgsz=IMGSZ)
    row = pd.DataFrame([dict(exp_id=exp, use_synth=use_synth, synth_ratio=1.0 if use_synth else 0, fold=f'f{fold}', seed=seed,
                             test_map50=r.box.map50, test_map50_95=r.box.map)])
    led = f'{D}/05_results/master_results_v2.csv'
    row.to_csv(led, mode='a', header=not os.path.exists(led), index=False)
    json.dump(dict(exp_id=exp, seed=seed, fold=fold, use_synth=use_synth, torch=torch.__version__), open(f'{RD}/run_info.json', 'w'))
    open(f'{RD}/DONE', 'w').write('ok')

def ensure_detectors(cfg, seed):
    for fold in range(4):
        if os.path.exists(f'{D}/04_runs/{exp_name(cfg, fold, seed)}/DONE'): continue
        assert TRAIN_MISSING, f'missing detector {exp_name(cfg, fold, seed)} and TRAIN_MISSING=False'
        if time.time() - T0 > MAX_TRAIN_HOURS * 3600:
            print(f'training budget reached at fold {fold}; re-run this cell to continue'); return False
        print('training', exp_name(cfg, fold, seed)); train_one(cfg, fold, seed)
    return True

def get_preds(cfg, seed):
    path = f'{D}/05_results/oof_preds_{cfg}_s{seed}.csv'
    if os.path.exists(path): return pd.read_csv(path, dtype={'stem': str})
    out = []
    for k in range(4):
        mdl = YOLO(f'{D}/04_runs/{exp_name(cfg, k, seed)}/weights/best.pt')
        stems = list(man[man.fold == k].image_id)
        for r in mdl.predict([str(Y/'images'/f'{s}.jpg') for s in stems], conf=0.01, iou=0.6, imgsz=640, max_det=100, stream=True, verbose=False, batch=16):
            for (x1, y1, x2, y2), cf, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy()):
                out.append((Path(r.path).stem, k, int(c), float(cf), x1, y1, x2, y2))
    p = pd.DataFrame(out, columns=['stem', 'fold', 'cls', 'conf', 'x1', 'y1', 'x2', 'y2']); p.to_csv(path, index=False); return p

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


## 3. Features, reference-free head, evaluation

In [9]:
def engine_feats(df):
    out = np.zeros((len(df), 4)); xy = df[['x1', 'y1', 'x2', 'y2']].values
    for stem, sub in df.groupby('stem'):
        mt, mx = masks(stem); a = prep(mt, mx, ENG[0], ENG[1])
        for i in sub.index: out[i] = query2(a, xy[i], ENG[2], ENG[3])
    return out

def make_crops(df):
    X = np.zeros((len(df), CROP, CROP), np.uint8); h = CROP // 2
    cx = np.clip(((df.x1 + df.x2) / 2).values.astype(int), 0, 639); cy = np.clip(((df.y1 + df.y2) / 2).values.astype(int), 0, 639)
    for stem, sub in df.groupby('stem'):
        gp = np.pad(cv2.imread(str(Y/'images'/f'{stem}.jpg'), 0), h, mode='reflect')
        for i in sub.index: X[i] = gp[cy[i]:cy[i] + CROP, cx[i]:cx[i] + CROP]
    return X

import torch.nn as nn
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
def make_net():
    blk = lambda i, o: [nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU()]
    return nn.Sequential(*blk(1, 16), nn.MaxPool2d(2), *blk(16, 32), nn.MaxPool2d(2), *blk(32, 64),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 3))

def train_head(X, y, seed=0):
    torch.manual_seed(seed); net = make_net().to(dev); opt = torch.optim.Adam(net.parameters(), 2e-3)
    cnt = np.bincount(y, minlength=3) + 1; w = torch.tensor((cnt.sum() / cnt / 3).clip(0.2, 5), dtype=torch.float32, device=dev)
    lossf = nn.CrossEntropyLoss(weight=w); Xt = torch.tensor(X[:, None] / 255., dtype=torch.float32); yt = torch.tensor(y).long()
    for ep in range(HEAD_EPOCHS):
        net.train(); perm = torch.randperm(len(Xt))
        for i in range(0, len(perm), 256):
            idx = perm[i:i + 256]; xb = Xt[idx].to(dev); yb = yt[idx].to(dev)
            if np.random.rand() < .5: xb = torch.flip(xb, [3])
            if np.random.rand() < .5: xb = torch.flip(xb, [2])
            opt.zero_grad(); lossf(net(xb), yb).backward(); opt.step()
    return net

def head_probs(net, X):
    net.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), 512):
            out.append(torch.softmax(net(torch.tensor(X[i:i + 512, None] / 255., dtype=torch.float32).to(dev)), 1).cpu().numpy())
    return np.concatenate(out)

def add_features(df, with_head=True):
    df['crit'] = df.cls.isin(CRIT).astype(int); df['conf_crit'] = df.conf * df.crit
    df['logarea'] = np.log((df.x2 - df.x1).clip(1) * (df.y2 - df.y1).clip(1))
    df['g_agree'] = (((df.cls == 0) & (df.g_open > 0)) | ((df.cls == 1) & (df.g_short > 0))).astype(int)
    df['g_cross'] = (((df.cls == 0) & (df.g_short > 0)) | ((df.cls == 1) & (df.g_open > 0))).astype(int)
    df['lm'] = np.log1p(df.missing); df['le'] = np.log1p(df.excess); df['conf_agree'] = df.conf * df.g_agree
    if with_head:
        df['p_agree'] = np.where(df.cls == 0, df.p_open, np.where(df.cls == 1, df.p_short, 0.0)); df['conf_pagree'] = df.conf * df.p_agree
    return df

F_IMG = ['conf', 'crit', 'conf_crit', 'logarea']
F_GRAPH = F_IMG + ['g_open', 'g_short', 'g_agree', 'g_cross', 'lm', 'le', 'conf_agree']
F_HEAD = F_IMG + ['p_open', 'p_short', 'p_agree', 'conf_pagree']
POL = {'A0 raw conf': None, 'A image-only': F_IMG, 'B +graph (template)': F_GRAPH, 'C +head (reference-free)': F_HEAD}

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def iou_mat(a, b):
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0]); iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2]); iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    A = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); B = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (A[:, None] + B[None, :] - inter + 1e-9)

def build_gt(df):
    groups = {s: sub.index.values for s, sub in df.groupby('stem')}
    y = np.zeros(len(df), int); rows = []
    for stem in man.image_id:
        g = GT[stem]; g = g[np.isin(g[:, 0], CRIT)]
        if len(g) == 0: continue
        idx = groups.get(stem, np.array([], int))
        I = iou_mat(df.loc[idx, ['x1', 'y1', 'x2', 'y2']].values, g[:, 1:5]) if len(idx) else np.zeros((0, len(g)))
        if len(idx): y[idx] = (I.max(1) >= 0.5).astype(int)
        for j in range(len(g)):
            rows.append(dict(stem=stem, board=board_of[stem], pred_idx=idx[I[:, j] >= 0.5] if len(idx) else np.array([], int)))
    df['y'] = y
    return pd.DataFrame(rows)

gt_scores = lambda sc, sub: np.array([sc[i].max() if len(i) else -1.0 for i in sub.pred_idx])

def tau_for(s, alpha):                               # split-conformal quantile of critical-defect scores
    s = np.sort(s); k = int(np.floor(alpha * (len(s) + 1)))
    return -np.inf if k < 1 else s[k - 1]

def lobo(df, gt):
    gtb = {b: gt[gt.board == b] for b in order}; rows = []
    for i, b in enumerate(order):
        calib = [order[(i + k) % 11] for k in (1, 2, 3)]; fit = [x for x in order if x != b and x not in calib]
        m_fit = df.board.isin(fit).values; mte = (df.board == b).values; n_img = int((man.source_board == b).sum())
        for pol, feats in POL.items():
            if feats is None: sc = (df.conf * df.crit).values
            else:
                mdl = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
                mdl.fit(df.loc[m_fit, feats].fillna(0), df.loc[m_fit, 'y']); sc = mdl.predict_proba(df[feats].fillna(0))[:, 1]
            s_by = [gt_scores(sc, gtb[c]) for c in calib]; s_te = gt_scores(sc, gtb[b])
            for al in ALPHAS:
                taus = {'pooled': tau_for(np.concatenate(s_by), al), 'worst-board': min(tau_for(s, al) for s in s_by)}
                for meth, tau in taus.items():
                    flag = mte & (sc >= tau)
                    rows.append(dict(policy=pol, method=meth, alpha=al, board=b, n_gt=len(s_te),
                                     n_miss=int(((s_te < tau) | (s_te == -1)).sum()), flagged=int(flag.sum()), n_img=n_img, tp=int(df.y[flag].sum())))
    return pd.DataFrame(rows)

def boot_summary(R, ref='A image-only'):
    rng = np.random.default_rng(1); out = []
    idx = rng.integers(0, 11, (N_BOOT, 11))
    for (al, meth), g in R.groupby(['alpha', 'method']):
        arr = {p: gg.set_index('board').loc[order] for p, gg in g.groupby('policy')}
        stat = lambda a: (a.n_miss.values[idx].sum(1) / a.n_gt.values[idx].sum(1), a.flagged.values[idx].sum(1) / a.n_img.values[idx].sum(1))
        mr, lr = stat(arr[ref])
        for p, a in arr.items():
            m, l = stat(a)
            out.append(dict(alpha=al, method=meth, policy=p, miss=a.n_miss.sum() / a.n_gt.sum(), miss_over_alpha=a.n_miss.sum() / a.n_gt.sum() / al,
                            load=a.flagged.sum() / a.n_img.sum(), precision=a.tp.sum() / max(1, a.flagged.sum()),
                            d_load=np.mean(l - lr), d_load_lo=np.percentile(l - lr, 2.5), d_load_hi=np.percentile(l - lr, 97.5),
                            d_miss=np.mean(m - mr), d_miss_lo=np.percentile(m - mr, 2.5), d_miss_hi=np.percentile(m - mr, 97.5)))
    return pd.DataFrame(out)

In [11]:
def run_cfg(cfg, seed):
    print(f'\n================ config={cfg} seed={seed} ================')
    if not ensure_detectors(cfg, seed): return None
    df = get_preds(cfg, seed).reset_index(drop=True); df['board'] = df.stem.map(board_of)
    df[['g_open', 'g_short', 'missing', 'excess']] = engine_feats(df)
    # reference-free head: out-of-fold, teacher = template-based engine verdict, input = test-image crop only
    X = make_crops(df); yt = np.array([verdict(a > 0, b > 0, c, d) for a, b, c, d in df[['g_open', 'g_short', 'missing', 'excess']].values])
    pr = np.zeros((len(df), 3))
    for h in range(4):
        tr, te = (df.fold != h).values, (df.fold == h).values
        net = train_head(X[tr], yt[tr]); pr[te] = head_probs(net, X[te])
        print(f'head fold {h}: agreement with teacher = {(pr[te].argmax(1) == yt[te]).mean():.3f}')
    df['p_open'], df['p_short'] = pr[:, 1], pr[:, 2]
    df = add_features(df); gt = build_gt(df)
    print(len(gt), 'critical GT defects;', int((gt.pred_idx.map(len) == 0).sum()), 'unmatched by any box')
    R = lobo(df, gt); S = boot_summary(R)
    tag = f'{cfg}_s{seed}'
    R.to_csv(f'{D}/05_results/lobo_{tag}.csv', index=False); S.to_csv(f'{D}/05_results/lobo_summary_{tag}.csv', index=False)
    return dict(df=df, gt=gt, R=R, S=S)

results = {}
for cfg in CONFIGS:
    for seed in SEEDS:
        r = run_cfg(cfg, seed)
        if r is not None: results[(cfg, seed)] = r
print('\nfinished configs:', list(results))


================ config=real seed=42 ================
head fold 0: agreement with teacher = 0.607
head fold 1: agreement with teacher = 0.792
head fold 2: agreement with teacher = 0.779
head fold 3: agreement with teacher = 0.710
3448 critical GT defects; 97 unmatched by any box

finished configs: [('real', 42)]


## 4. Results, bootstrap, automatic verdict

In [12]:
pd.set_option('display.width', 200)
for key, r in results.items():
    print(f'\n######## {key} ########')
    S = r['S']
    for (al, meth), g in S.groupby(['alpha', 'method']):
        print(f'\nalpha={al}  method={meth}')
        print(g[['policy', 'miss', 'miss_over_alpha', 'load', 'precision', 'd_load', 'd_load_lo', 'd_load_hi', 'd_miss', 'd_miss_lo', 'd_miss_hi']]
              .round(3).to_string(index=False))
    print('\n(d_* = difference vs "A image-only", mean and 95% bootstrap CI over the 11 boards; negative = better)')


######## ('real', 42) ########

alpha=0.05  method=pooled
                  policy  miss  miss_over_alpha  load  precision  d_load  d_load_lo  d_load_hi  d_miss  d_miss_lo  d_miss_hi
            A image-only 0.059            1.183 5.845      0.393   0.000      0.000      0.000   0.000      0.000      0.000
             A0 raw conf 0.059            1.172 6.394      0.357   0.648     -0.356      2.577  -0.001     -0.008      0.004
     B +graph (template) 0.058            1.160 4.132      0.573  -1.795     -3.708     -0.617  -0.001     -0.017      0.009
C +head (reference-free) 0.057            1.131 5.979      0.395   0.151     -0.289      0.668  -0.003     -0.012      0.002

alpha=0.05  method=worst-board
                  policy  miss  miss_over_alpha   load  precision  d_load  d_load_lo  d_load_hi  d_miss  d_miss_lo  d_miss_hi
            A image-only 0.037            0.731 11.765      0.211   0.000      0.000      0.000   0.000      0.000      0.000
             A0 raw conf 0.036  

In [13]:
def checklist(S, eng_heldout):
    rep = []; ok = lambda b: 'PASS' if b else 'FAIL'
    e = eng_heldout
    c1 = min(e['acc_open'], e['acc_short']) >= 0.85 and e['false_flag'] <= 0.15
    rep.append((f"1 Engine (held-out boards): strict acc open/short >= .85, false-flag <= .15  "
                f"[{e['acc_open']:.2f}/{e['acc_short']:.2f}, {e['false_flag']:.2f}]", c1))
    g = S[(S.method == 'worst-board') & (S.alpha == 0.10)].set_index('policy')
    def better(p):
        r = g.loc[p]
        return (r.d_load_hi < 0 and r.d_miss <= 0.005) or (r.d_miss_hi < 0 and r.d_load <= 0.1)
    rep.append(('2 B (graph) beats image-only at alpha=.10 (worst-board), CI excludes 0 on one axis, not worse on the other', better('B +graph (template)')))
    pooled = S[(S.method == 'pooled')].groupby('policy').miss_over_alpha.mean(); worst = S[(S.method == 'worst-board')].groupby('policy').miss_over_alpha.mean()
    rep.append((f"3 Board shift: worst-board conformal keeps miss <= 1.25*alpha for B  [pooled {pooled['B +graph (template)']:.2f}x, worst-board {worst['B +graph (template)']:.2f}x]",
                worst['B +graph (template)'] <= 1.25))
    rep.append(('4 Reference-free head C is not worse than template-based B (load diff <= .1, miss diff <= .01 vs B)',
                (g.loc['C +head (reference-free)'].load - g.loc['B +graph (template)'].load) <= 0.1 and
                (g.loc['C +head (reference-free)'].miss - g.loc['B +graph (template)'].miss) <= 0.01))
    return rep

report = {}
for key, r in results.items():
    rep = checklist(r['S'], dict(acc_open=ao, acc_short=as_, false_flag=ff))
    print(f'\n== checklist {key} ==')
    for t, b in rep: print(('PASS  ' if b else 'FAIL  ') + t)
    report[f'{key[0]}_s{key[1]}'] = [dict(check=t, passed=bool(b)) for t, b in rep]
json.dump(report, open(f'{D}/05_results/verdict_report.json', 'w'), indent=1)
print('\nPASS on 1-3, repeated across seeds, is evidence worth examining further (and bringing to a patent agent); FAIL tells you what to fix or drop. It is not a patentability opinion.')


== checklist ('real', 42) ==
FAIL  1 Engine (held-out boards): strict acc open/short >= .85, false-flag <= .15  [0.64/0.74, 0.01]
PASS  2 B (graph) beats image-only at alpha=.10 (worst-board), CI excludes 0 on one axis, not worse on the other
PASS  3 Board shift: worst-board conformal keeps miss <= 1.25*alpha for B  [pooled 1.09x, worst-board 0.69x]
FAIL  4 Reference-free head C is not worse than template-based B (load diff <= .1, miss diff <= .01 vs B)

PASS on 1-3, repeated across seeds, is evidence worth examining further (and bringing to a patent agent); FAIL tells you what to fix or drop. It is not a patentability opinion.


In [14]:
# detector comparison: real vs synthetic (needs both configs trained)
led = pd.read_csv(f'{D}/05_results/master_results_v2.csv')
led = led[led.fold.astype(str).str.startswith('f')]
if led.use_synth.nunique() == 2:
    pv = led.pivot_table(index=['fold', 'seed'], columns='use_synth', values='test_map50_95', aggfunc='mean').dropna()
    d = pv[True] - pv[False]
    print(pv.round(4)); print(f'synth - real (mAP50-95): {d.mean():+.4f} +- {d.std():.4f}; synth better in {(d > 0).sum()}/{len(d)} fold-seeds')
else:
    print('only one detector type trained so far; set CONFIGS = ["real", "synth1"] to compare')

use_synth    False   True 
fold  seed                
f0    42    0.5998  0.6342
f1    42    0.6340  0.6268
f2    42    0.6474  0.6835
f3    42    0.7348  0.7109
fixed 42    0.6951  0.7443
synth - real (mAP50-95): +0.0177 +- 0.0315; synth better in 3/5 fold-seeds
